In [ ]:

import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition as TC
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

## Data Loading

In [ ]:

file_name = 'data/world_cup_2026_input.xlsx'
df = pd.read_excel(file_name, sheet_name='Master + Ranking', index_col=None, header=None)

#designating variables for the Master + Ranking sheet
team   = df.iloc[1:49, 1].reset_index(drop=True) #team name
confed = df.iloc[1:49, 4].reset_index(drop=True)  #confederation
pot    = df.iloc[1:49, 7].reset_index(drop=True)  #pot
points = pd.to_numeric(df.iloc[1:49, 6], errors='coerce').reset_index(drop=True)  #FIFA points
fi     = pd.to_numeric(df.iloc[1:49, 12], errors='coerce').reset_index(drop=True) #spectator index




# Six playoff nations (4 UEFA + 2 Inter-Continental) are unconfirmed at draw time.
# We replace their NaN points with the mean of confirmed teams in the SAME pot.
# Pot-average imputation is fairer than a global mean: UEFA playoff winners sit in Pot 4 but are much stronger than typical Pot-4 nations, so using the Pot-4 average

for i in points.index:
    if pd.isna(points[i]):
        vals = [points[k] for k in points.index if pot[k] == pot[i] and not pd.isna(points[k])]
        points[i] = float(np.mean(vals)) if vals else 0.0  


# The two Inter-Continental playoff winners have no confirmed nationality, so their fi* cannot be computed from the distance formula.  We substitute the mean fi*
# of all confirmed teams in the same confederation group the best available proxy.
# fi[i]==0 is also treated as missing: a zero spectator index would incorrectly suppress those matches' popularity to near zero in Q3.
for i in fi.index:
    if pd.isna(fi[i]) or fi[i] == 0:
        vals = [fi[k] for k in fi.index
                if confed[k] == confed[i] and not pd.isna(fi[k]) and fi[k] > 0]
        fi[i] = float(np.mean(vals)) if vals else 0.0

# pot4_avg: used only in Figure 1 to approximate points for unconfirmed playoff
pot4_avg = float(np.mean([points[i] for i in range(len(team)) if pot[i] == 'Pot 4']))

#  GroupSeats sheet 
dg    = pd.read_excel(file_name, sheet_name='GroupSeats', index_col=0)
group = list(dg.index)   
S     = dg['Avg Offered Seats']   

# Official Allocation sheet 
# 72 rows = 12 letters × 6 matches each. 
da         = pd.read_excel(file_name, sheet_name='Official Allocation', index_col=None, header=None)
da.columns = ['RS', 'Match', 'Stadium', 'Capacity']  
da         = da.iloc[1:].copy()          
da['Match'] = da['Match'].astype(int)   

#  Stadiums sheet 
ds = pd.read_excel(file_name, sheet_name='Stadiums')
ds.columns = [c.strip() for c in ds.columns]        
ds = ds[ds['Stadium (FIFA)'].notna()].copy()          
ds['Group-stage matches'] = ds['Group-stage matches'].astype(int)  

# Actual Draw sheet 
# The real FIFA draw result.
dact = pd.read_excel(file_name, sheet_name='Actual Draw', header=None)

print("Data loaded successfully.")
print(f"Teams: {len(team)}  |  Groups: {len(group)}  |  Stadiums: {len(ds)}")

## Q2 — Group Formation Model

In [ ]:
# Q2: Group Formation Model 
teams  = list(team)    
groups = list(group)  

# Scalar dictionaries for fast lookup 
p   = {team[i]: points[i] for i in range(len(team))}   # FIFA ranking points per nation
fiv = {team[i]: fi[i]     for i in range(len(team))}   # fi* spectator index per nation

# Partition teams by pot and confederation for use in constraint rules.
pots     = sorted(set(pot))
pot_set  = {p_: [team[i] for i in range(len(team)) if pot[i] == p_]  for p_ in pots}
confs    = sorted(set(confed))
conf_set = {c:  [team[i] for i in range(len(team)) if confed[i] == c] for c in confs}

# non_uefa: confederations that get the strict <=1-per-group cap.
non_uefa = [c for c in confs if c not in ('UEFA', 'Inter-Continental')]

#  Model
model = pyo.ConcreteModel()

# Decision var: x[i,j] = 1 if nation i is placed in group j, else 0.
model.x = pyo.Var(teams, groups, domain=pyo.Binary)

# Auxiliary variable: w = the minimum group FIFA-points total across all 12 groups.
model.w = pyo.Var(domain=pyo.NonNegativeReals)

# Objective: maximise w = min_j(sum_i p[i]*x[i,j])
model.obj = pyo.Objective(expr=model.w, sense=pyo.maximize)

# Constraint w: w <= total points in every group j.
def rule_w(model, j):
    return model.w <= sum(p[i] * model.x[i, j] for i in teams)
model.constr_w = pyo.Constraint(groups, rule=rule_w)

# Constraint 1
def rule1(model, i):
    return sum(model.x[i, j] for j in groups) == 1
model.constr1 = pyo.Constraint(teams, rule=rule1)

# Constraint 2
def rule2(model, j):
    return sum(model.x[i, j] for i in teams) == 4
model.constr2 = pyo.Constraint(groups, rule=rule2)

# Constraint 3
def rule3(model, j, p_):
    return sum(model.x[i, j] for i in pot_set[p_]) == 1
model.constr3 = pyo.Constraint(groups, pots, rule=rule3)

# Constraint 4
def rule4(model, j, c):
    return sum(model.x[i, j] for i in conf_set[c]) <= 1
model.constr4 = pyo.Constraint(groups, non_uefa, rule=rule4)

# Constraint 5
# 16 UEFA teams across 12 groups: exactly 4 groups will contain 2 UEFA teams.
def rule5(model, j):
    return sum(model.x[i, j] for i in conf_set['UEFA']) <= 2
model.constr5 = pyo.Constraint(groups, rule=rule5)

# Host-nation fixing: aligned to official FIFA draw for direct comparison.
# Mexico->A, Canada->B, USA->D 
model.fix_mexico = pyo.Constraint(expr=model.x['Mexico',        'A'] == 1)
model.fix_canada = pyo.Constraint(expr=model.x['Canada',        'B'] == 1)
model.fix_usa    = pyo.Constraint(expr=model.x['United States', 'D'] == 1)

# Solver
solver = pyo.SolverFactory('glpk')
results = solver.solve(
    model,
    load_solutions=False,
    options={
        'mipgap': 0.001,   
        'tmlim':  300     
    }
)


from pyomo.opt import TerminationCondition as TC
if results.solver.status == SolverStatus.ok and results.solver.termination_condition in (
        TC.optimal, TC.maxTimeLimit, TC.feasible, TC.other):
    model.solutions.load_from(results)
else:
    raise RuntimeError(f"Q2 solve failed: {results.solver.termination_condition}")

# Extract solution 

xv = pd.DataFrame(index=groups, columns=teams, dtype=float)
for i in teams:
    for j in groups:
        v = pyo.value(model.x[i, j])
        xv.loc[j, i] = v if v is not None else 0.0
xv = xv.fillna(0)

oval = pyo.value(model.obj)   

for j in groups:
    members = [i for i in teams if xv.loc[j, i] > 0.5]
    print(f"Group {j}: {', '.join(members)}  |  pts: {sum(p[i] for i in members):.1f}")
print(f"Objective w (min group pts) = {oval:.2f}")

In [ ]:
# Figure 1: Actual FIFA Draw vs Optimised Group Formation
playoff_map = {
    'UEFA PO A Winner': 'UEFA Playoff A Winner',
    'UEFA PO B Winner': 'UEFA Playoff B Winner',
    'UEFA PO C Winner': 'UEFA Playoff C Winner',
    'UEFA PO D Winner': 'UEFA Playoff D Winner',
    'IC PO 1 Winner':   'Inter-Continental Playoff 1 Winner',
    'IC PO 2 Winner':   'Inter-Continental Playoff 2 Winner',
}


actual_groups = {}
for _, row in dact.iloc[1:].iterrows():
    g = str(row[0]).strip()
    if g in ('None', 'nan', ''):
        continue
    members = []
    for c in range(1, 5):
        name = str(row[c]).strip()
        name = playoff_map.get(name, name)   
        members.append(name)
    actual_groups[g] = members

# Sum FIFA points per group; unconfirmed playoff nations get pot4_avg as best estimate.

actual_pts = []
for g in actual_groups:
    total = 0
    for t in actual_groups[g]:
        total = total + p.get(t, pot4_avg)
    actual_pts.append(total)
actual_pts.sort(reverse=True)

model_pts = []
for j in groups:
    total = 0
    for i in teams:
        if xv.loc[j, i] > 0.5:
            total = total + p[i]
    model_pts.append(total)
model_pts.sort(reverse=True)


y_min = min(actual_pts + model_pts) - 200
y_max = max(actual_pts + model_pts) + 200

x_labels = ['1','2','3','4','5','6','7','8','9','10','11','12']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))


axes[0].bar(x_labels, actual_pts, color='red', edgecolor='black', linewidth=0.5)
axes[0].axhline(np.mean(actual_pts), color='black', linestyle='--', linewidth=1.5,
                label='Mean: ' + str(round(np.mean(actual_pts))))
axes[0].set_ylim(y_min, y_max)   
axes[0].set_xlabel('Group rank (1 = strongest)')
axes[0].set_ylabel('Total FIFA Points')
axes[0].set_title('Actual Draw  (Spread: ' + str(round(max(actual_pts)-min(actual_pts))) + ' pts)',
                  fontweight='bold')
axes[0].legend()


axes[1].bar(x_labels, model_pts, color='blue', edgecolor='black', linewidth=0.5)
axes[1].axhline(np.mean(model_pts), color='red', linestyle='--', linewidth=1.5,
                label='Mean: ' + str(round(np.mean(model_pts))))
axes[1].set_ylim(y_min, y_max)   
axes[1].set_xlabel('Group rank (1 = strongest)')
axes[1].set_ylabel('Total FIFA Points')
axes[1].set_title('Optimised  (Spread: ' + str(round(max(model_pts)-min(model_pts))) + ' pts)',
                  fontweight='bold')
axes[1].legend()

fig.suptitle('Figure 1: Actual Draw vs Optimised Group Formation', fontweight='bold')
plt.tight_layout()
plt.show()
plt.close('all')


## Q3 — Group-Letter Assignment Model

In [ ]:
# Q3: Group-Letter Assignment Model 


host_groups, other_groups = {}, []
for j in groups:
    members = [i for i in teams if xv.loc[j, i] > 0.5]
    host = next((m for m in members if m in ['United States', 'Mexico', 'Canada']), None)
    if   host == 'Mexico':        host_groups[1] = members
    elif host == 'Canada':        host_groups[2] = members
    elif host == 'United States': host_groups[3] = members
    else: other_groups.append(members)

other_groups.sort(key=lambda m: sum(p[t] for t in m), reverse=True)  
q2g = {**host_groups, **{i + 4: m for i, m in enumerate(other_groups)}}

#  Rank teams within each group 

for k in q2g:
    host = next((m for m in q2g[k] if m in ['United States', 'Mexico', 'Canada']), None)
    if host:
        q2g[k] = [host] + sorted([m for m in q2g[k] if m != host],
                                   key=lambda t: p[t], reverse=True)
    else:
        q2g[k] = sorted(q2g[k], key=lambda t: p[t], reverse=True)

#  Match popularity function
def mpop(fi1, fi2):
    f_hat   = (fi1 + fi2) / 2     
    f_tilde = 1.0 if max(fi1, fi2) >= 1.0 else (fi1 + fi2) / 2
    # f_tilde = 1 when at least one team has fi*=1 (guaranteed full house of their fans);
    # officials and neutral fans also fill their allocation => contributes 1 full unit.
    # Otherwise officials attend at the same average rate as other neutral fans.
    return fi1 + fi2 + f_hat + f_tilde  

# Rank-pair mapping and precomputed match popularities
rp = {1: (1,2), 2: (1,3), 3: (1,4), 4: (2,3), 5: (2,4), 6: (3,4)}

# pm[k][(q1,q2)]: popularity of the match between rank-q1 and rank-q2 in group k.
# Precomputed outside the model as CONSTANTS: pm values depend only on team identities
# Keeping them as constants avoids nonlinearity in the model's row-set constraint (C4).
pm = {}
for k in q2g:
    pm[k] = {rp[mn]: mpop(fiv[q2g[k][rp[mn][0] - 1]], fiv[q2g[k][rp[mn][1] - 1]])
             for mn in rp}

# Schedule lookup from Official Allocation sheet
sched = {}
for _, row in da.iterrows():
    l, mn, s = row['RS'], int(row['Match']), row['Stadium']
    if l not in sched:
        sched[l] = {}
    sched[l][mn] = s

# slots[s] = list of (letter, rank_pair) for every match slot hosted at stadium s.
# If group k is assigned to letter l, then slot (l, rp_) at stadium s contributes
stad  = da['Stadium'].unique().tolist()   
slots = {s: [] for s in stad}
for l, d in sched.items():
    for mn, s in d.items():
        slots[s].append((l, rp[mn]))      

# Model parameters
grp_idx = list(q2g.keys())   
letters = groups              
BIG_M = len(stad) * 4.0   

# Model 
model_q3 = pyo.ConcreteModel()

# z[k,l] = 1 if group k is assigned to letter l, else 0.
model_q3.z    = pyo.Var(grp_idx, letters, domain=pyo.Binary)
model_q3.y    = pyo.Var(stad, domain=pyo.NonNegativeReals)

# wmin: minimum row-set popularity across all 16 stadiums
model_q3.wmin = pyo.Var(domain=pyo.NonNegativeReals)

# wmax: maximum row-set popularity across all 16 stadiums 
model_q3.wmax = pyo.Var(domain=pyo.NonNegativeReals)

model_q3.v    = pyo.Var(stad, domain=pyo.Binary)

# OBJECTIVE
model_q3.obj = pyo.Objective(expr=model_q3.wmin + model_q3.wmax, sense=pyo.maximize)

# Constraint C1
def rule_c1(model, k):
    return sum(model.z[k, l] for l in letters) == 1
model_q3.c1 = pyo.Constraint(grp_idx, rule=rule_c1)

# Constraint C2
def rule_c2(model, l):
    return sum(model.z[k, l] for k in grp_idx) == 1
model_q3.c2 = pyo.Constraint(letters, rule=rule_c2)

# Host-nation fixing: Mexico->A, Canada->B, USA->D 
model_q3.fix_A = pyo.Constraint(expr=model_q3.z[1, 'A'] == 1)
model_q3.fix_B = pyo.Constraint(expr=model_q3.z[2, 'B'] == 1)
model_q3.fix_D = pyo.Constraint(expr=model_q3.z[3, 'D'] == 1)

# Constraint C4
def rule_c4(model, s):
    return model.y[s] == sum(pm[k][rp_] * model.z[k, l]
                             for (l, rp_) in slots[s] for k in grp_idx)
model_q3.c4 = pyo.Constraint(stad, rule=rule_c4)

# Constraint C5
def rule_c5(model, s):
    return model.wmin <= model.y[s]
model_q3.c5 = pyo.Constraint(stad, rule=rule_c5)

# Constraint C6
def rule_c6(model, s):
    return model.wmax >= model.y[s]
model_q3.c6 = pyo.Constraint(stad, rule=rule_c6)

# Constraint C7
def rule_c7(model, s):
    return model.wmax <= model.y[s] + BIG_M * (1 - model.v[s])
model_q3.c7 = pyo.Constraint(stad, rule=rule_c7)

# Constraint C8
model_q3.c8 = pyo.Constraint(expr=sum(model_q3.v[s] for s in stad) == 1)

# Solve 
solver_q3  = pyo.SolverFactory('glpk')
results_q3 = solver_q3.solve(model_q3, load_solutions=False, tee=False)
if (results_q3.solver.status == SolverStatus.ok and
        results_q3.solver.termination_condition == TC.optimal):
    model_q3.solutions.load_from(results_q3)
else:
    raise RuntimeError("Q3 solve failed.")

#  Extract solution 

asgn = {}
for k in grp_idx:
    for l in letters:
        if pyo.value(model_q3.z[k, l]) > 0.5:   
            asgn[l] = k

yv       = {s: pyo.value(model_q3.y[s]) for s in stad}   
wmin_val = pyo.value(model_q3.wmin)   
wmax_val = pyo.value(model_q3.wmax)   

print(f"Q3 Optimal:  w_min={wmin_val:.4f}   w_max={wmax_val:.4f}")
print(f"Objective (w_min + w_max) = {wmin_val + wmax_val:.4f}")
print()
for l in letters:
    k = asgn[l]
    print(f"Letter {l}: Group {k}  ->  {', '.join(q2g[k])}")
print()

for s in sorted(stad, key=lambda s: yv[s], reverse=True):
    tag = "  <- w_max (binding)" if abs(yv[s] - wmax_val) < 1e-4 else           ("  <- w_min (binding)" if abs(yv[s] - wmin_val) < 1e-4 else "")
    print(f"  {s:<42}  y = {yv[s]:.4f}{tag}")

In [ ]:
#  Figure 2
# Map each actual FIFA group letter to the closest Q2 group by team overlap.
def best_q2(act_members):
    best_k = None
    best_n = -1
    for k in q2g:
        n = 0
        for m in act_members:
            if m in q2g[k]:
                n = n + 1
        if n > best_n:
            best_n = n
            best_k = k
    return best_k

act_ltr_k = {}
for l in actual_groups:
    act_ltr_k[l] = best_q2(actual_groups[l])

# Compute actual row-set popularity per stadium using the Official Allocation schedule.
yv_act = {}
for s in stad:
    total = 0
    for (l, rp_) in slots[s]:
        if l in act_ltr_k:
            total = total + pm[act_ltr_k[l]][rp_]
    yv_act[s] = total


stad_pairs = []
for s in stad:
    stad_pairs.append((yv[s], s))
stad_pairs.sort(reverse=True)
sorted_stads = []
for val, s in stad_pairs:
    sorted_stads.append(s)
short = []
for s in sorted_stads:
    short.append(s.replace(' Stadium', ''))

x_pos = np.arange(len(sorted_stads))
bw    = 0.38

fig, ax = plt.subplots(figsize=(16, 5))

ax.bar(x_pos - bw/2, [yv_act[s] for s in sorted_stads], width=bw,
       label='Actual FIFA', color='red', alpha=0.8, edgecolor='black', linewidth=0.4)
ax.bar(x_pos + bw/2, [yv[s]     for s in sorted_stads], width=bw,
       label='Optimal Q3', color='blue', alpha=0.8, edgecolor='black', linewidth=0.4)


ax.axhline(wmax_val,             color='blue', linestyle='--', linewidth=1.5,
           label='w_max opt=' + str(round(wmax_val, 3)))
ax.axhline(wmin_val,             color='blue', linestyle=':',  linewidth=1.5,
           label='w_min opt=' + str(round(wmin_val, 3)))
ax.axhline(max(yv_act.values()), color='red',  linestyle='--', linewidth=1.5,
           label='w_max act=' + str(round(max(yv_act.values()), 3)))
ax.axhline(min(yv_act.values()), color='red',  linestyle=':',  linewidth=1.5,
           label='w_min act=' + str(round(min(yv_act.values()), 3)))

ax.set_xticks(x_pos)
ax.set_xticklabels(short, rotation=35, ha='right', fontsize=8)
ax.set_xlabel('Stadium (row-set)')
ax.set_ylabel('Row-Set Popularity (sum of match popularities)')
ax.set_title('Figure 2: Row-Set Popularity per Stadium — Actual FIFA vs Optimal',
             fontweight='bold')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()
plt.close('all')

print('Optimal: w_min=' + str(round(wmin_val,4)) + '  w_max=' + str(round(wmax_val,4)))
print('Actual:  w_min=' + str(round(min(yv_act.values()),4)) + '  w_max=' + str(round(max(yv_act.values()),4)))


In [ ]:
#  Q3 Results Table 
rows = []
for l in letters:
    k = asgn[l]                            
    for mn, pair in rp.items():            
        s = sched[l][mn]                   
        rows.append({
            'Letter':    l,
            'Group':     k,
            'Team 1':    q2g[k][pair[0] - 1],   
            'Team 2':    q2g[k][pair[1] - 1],   
            'Stadium':   s,
            'Match Pop': round(pm[k][pair], 4),  
            'RS Pop':    round(yv[s], 4),         
        })
q3_df = pd.DataFrame(rows)
print(q3_df.to_string(index=False))
print(f"\nTotal match popularity (72 matches): {q3_df['Match Pop'].sum():.4f}")

## Q4 — Stadium Assignment Model

In [ ]:
# Q4 Stadium Assignment 

# Row-set popularity per letter
# Sum of the 6 individual match popularities for each group letter.
rs_pop = {}
for l in letters:
    k = asgn[l]
    rs_pop[l] = sum(pm[k][rp[mn]] for mn in range(1, 7))

print("Row-set popularity per letter (Q3 output):")
for l in sorted(rs_pop, key=rs_pop.get, reverse=True):
    print(f"  Letter {l}: {rs_pop[l]:.4f}")
print()

#Stadium data 
cap   = dict(zip(ds['Stadium (FIFA)'], ds['Capacity']))
gsm   = dict(zip(ds['Stadium (FIFA)'], ds['Group-stage matches']))
stads = ds['Stadium (FIFA)'].tolist()
assert sum(gsm.values()) == 72

L_matches = {l: 6 for l in letters}

#model 
model_q4 = pyo.ConcreteModel()

model_q4.i   = pyo.Set(initialize=letters)
model_q4.j   = pyo.Set(initialize=stads)
model_q4.rsp = pyo.Param(model_q4.i, initialize=rs_pop)
model_q4.c   = pyo.Param(model_q4.j, initialize=cap)
model_q4.n   = pyo.Param(model_q4.j, initialize=gsm)
model_q4.m   = pyo.Param(model_q4.i, initialize=L_matches)

# x[i,j] = number of matches from letter i assigned to stadium j
model_q4.x = pyo.Var(model_q4.i, model_q4.j, domain=pyo.NonNegativeIntegers)

# Objective: capacity x row-set popularity x matches assigned
def obj_rule(model):
    return sum(model.rsp[i] * model.c[j] * model.x[i, j]
               for i in model.i for j in model.j)
model_q4.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

# Constraint 1
def rule_letter(model, i):
    return sum(model.x[i, j] for j in model.j) == model.m[i]
model_q4.c1 = pyo.Constraint(model_q4.i, rule=rule_letter)

# Constraint 2
def rule_stadium(model, j):
    return sum(model.x[i, j] for i in model.i) == model.n[j]
model_q4.c2 = pyo.Constraint(model_q4.j, rule=rule_stadium)

# Solve 
solver_q4  = pyo.SolverFactory('glpk')
results_q4 = solver_q4.solve(model_q4, load_solutions=False, tee=False)
if (results_q4.solver.status == SolverStatus.ok and
        results_q4.solver.termination_condition == TC.optimal):
    model_q4.solutions.load_from(results_q4)
else:
    raise RuntimeError("solve failed.")

opt_total = pyo.value(model_q4.obj)

# official FIFA allocation
off_total = 0.0
for l in letters:
    for mn in range(1, 7):
        s_off = sched[l][mn]
        off_total += cap[s_off] * rs_pop[l]

#Extract results
q4_rows = []
for i in model_q4.i:
    for j in model_q4.j:
        x_val = int(round(pyo.value(model_q4.x[i, j])))
        if x_val > 0:
            q4_rows.append({
                'Letter':        i,
                'Stadium':       j,
                'Capacity':      cap[j],
                'Matches':       x_val,
                'RS Popularity': round(rs_pop[i], 4),
                'Pop Value':     round(rs_pop[i] * cap[j] * x_val, 2),
            })

xv4 = pd.DataFrame(q4_rows).sort_values(['RS Popularity', 'Capacity'],
                                          ascending=[False, False])

print(f"Q4 Optimised total:  {opt_total:,.2f}")
print(f"Official FIFA total: {off_total:,.2f}")
print(f"Improvement: {opt_total - off_total:,.2f}  ({(opt_total/off_total - 1)*100:.2f}%)")
print()
print(xv4.to_string(index=False))


In [ ]:
# Figure 3 Stadium Capacity and Row-Set Popularity
stad_pairs = [(yv[s], s) for s in stad]
stad_pairs.sort(reverse=True)
stad_sorted   = [s for _, s in stad_pairs]
short_names   = [s.replace(' Stadium', '') for s in stad_sorted]
x_pos = np.arange(len(stad_sorted))

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x_pos, [cap[s] for s in stad_sorted], width=0.5,
       color='blue', alpha=0.7, edgecolor='black', linewidth=0.4,
       label='Stadium Capacity')
ax2 = ax.twinx()
ax2.plot(x_pos, [yv[s]     for s in stad_sorted], color='green', linewidth=2,
         marker='o', markersize=6, label='Row-Set Popularity (Optimal)')
ax2.plot(x_pos, [yv_act[s] for s in stad_sorted], color='red',   linewidth=1.5,
         linestyle='--', marker='^', markersize=5, alpha=0.8,
         label='Row-Set Popularity (Actual)')
ax.set_xticks(x_pos)
ax.set_xticklabels(short_names, rotation=40, ha='right', fontsize=8)
ax.set_xlabel('Stadium (sorted by optimal row-set popularity)')
ax.set_ylabel('Stadium Capacity', color='blue')
ax2.set_ylabel('Row-Set Popularity', color='black')
ax.tick_params(axis='y', labelcolor='blue')
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labs1 + labs2, fontsize=9, loc='upper right')
ax.set_title('Figure 3: Stadium Capacity vs Row-Set Popularity — Optimised vs Actual',
             fontweight='bold')
plt.tight_layout()
plt.show()
plt.close('all')

# Figure 4Scatter — Row-Set Popularity vs Capacity 
opt_pop_vals = []
opt_cap_vals = []
for _, row in xv4.iterrows():
    for _ in range(int(row['Matches'])):
        opt_pop_vals.append(row['RS Popularity'])
        opt_cap_vals.append(row['Capacity'])

off_pop_vals = []
off_cap_vals = []
for l in letters:
    for mn in range(1, 7):
        s_off = sched[l][mn]
        off_pop_vals.append(rs_pop[l])
        off_cap_vals.append(cap[s_off])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(opt_pop_vals, opt_cap_vals, color='blue', alpha=0.7,
                edgecolors='black', linewidth=0.4, s=60)
z = np.polyfit(opt_pop_vals, opt_cap_vals, 1)
xl = np.linspace(min(opt_pop_vals), max(opt_pop_vals), 100)
axes[0].plot(xl, np.polyval(z, xl), 'k--', linewidth=1.5)
corr_opt = np.corrcoef(opt_pop_vals, opt_cap_vals)[0, 1]
axes[0].set_xlabel('Row-Set Popularity (Letter)')
axes[0].set_ylabel('Stadium Capacity')
axes[0].set_title('Optimised: Capacity vs Row-Set Popularity', fontweight='bold')
axes[0].text(0.05, 0.92, 'r = ' + str(round(corr_opt, 3)),
             transform=axes[0].transAxes, fontsize=10)

axes[1].scatter(off_pop_vals, off_cap_vals, color='red', alpha=0.7,
                edgecolors='black', linewidth=0.4, s=60)
z2 = np.polyfit(off_pop_vals, off_cap_vals, 1)
xl2 = np.linspace(min(off_pop_vals), max(off_pop_vals), 100)
axes[1].plot(xl2, np.polyval(z2, xl2), 'k--', linewidth=1.5)
corr_off = np.corrcoef(off_pop_vals, off_cap_vals)[0, 1]
axes[1].set_xlabel('Row-Set Popularity (Letter)')
axes[1].set_ylabel('Stadium Capacity')
axes[1].set_title('Official FIFA: Capacity vs Row-Set Popularity', fontweight='bold')
axes[1].text(0.05, 0.92, 'r = ' + str(round(corr_off, 3)),
             transform=axes[1].transAxes, fontsize=10)

fig.suptitle('Figure 4: Stadium Capacity vs Row-Set Popularity — Optimised vs Official',
             fontweight='bold')
plt.tight_layout()
plt.show()
plt.close('all')

print('A higher positive correlation indicates that higher-popularity letters are assigned to larger stadiums.')
print('Optimised total popularity value:     ' + str(round(opt_total, 2)))
print('Official FIFA total popularity value: ' + str(round(off_total, 2)))
diff = opt_total - off_total
pct  = round((opt_total / off_total - 1) * 100, 2)
print('Improvement: ' + str(round(diff, 2)) + '  (' + str(pct) + '%)')
